In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('E:/DBS_ansys_2a&3a.fld', sep=r'\s+', comment='#')


filepath = "E:/DBS_ansys_2a&3a.fld"
with open(filepath, "r") as f:
    lines = f.readlines()
    print(lines[0:10])
    
    data_lines = lines[3:100]
    cleaned = "\n".join(
        re.sub(r"\bNan\b", "nan", line, flags=re.IGNORECASE) for line in data_lines
    )
    data = np.fromstring(cleaned, sep=" ")
    data = data.reshape(-1, 6)
    print(data[0:20])
    data[:, :3] = data[:, :3]*1000
    print(data[0:20])
    data[:, :3] = np.rint(data[:, :3])
    print(data[0:20])
    

C:\Users\nebula\AppData\Local\Temp\ipykernel_13812\3772715819.py:5: DtypeWarning: Columns (0: Grid, 1: Output, 2: Min:) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('E:/DBS_ansys_2a&3a.fld', sep=r'\s+', comment='#')


['Grid Output Min: [-52mm -111mm -52mm] Max: m 54mm 71mm] Grid Size: m 1mm 1mm] \n', 'X, Y, Z, Vector data "<Jx,Jy,Jz>"\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -5.2000000000000005e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -5.1000000000000004e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -5.0000000000000003e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.9000000000000002e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.8000000000000001e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.7000000000000007e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.6000000000000006e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.5000000000000005e-02  Nan Nan Nan\n']
[[-0.052 -0.111 -0.051    nan    nan    nan]
 [-0.052 -0.111 -0.05     nan    nan    nan]
 [-0.052 -0.111 -0.049    nan    nan    nan]
 [-0.052 -0.111 -0.04

In [ ]:
import re
import numpy as np
 
MU0 = 4 * np.pi * 1e-7  # H/m
 
 
# ---------------------------------------------------------------------
# 1. Parse the .fld file: header (grid bounds/spacing) + data rows
# ---------------------------------------------------------------------
def parse_fld(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    print('total lines in file=',len(lines))
    header1 = lines[0]  # "Grid Output Min: [...] Max: [...]"
    header2 = lines[1]  # "Grid Size: [...] Unit: "mm""
 
    nums = lambda s: [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)]
 
    min_max_vals = nums(header1)
    grid_min = np.array(min_max_vals[0:3])   # in mm
    grid_max = np.array(min_max_vals[3:6])   # in mm
 
    size_vals = nums(header2)
    grid_size = np.array(min_max_vals[6:9])     # spacing in mm

    print('line 1779117',lines[1779117])
 
    unit_match = re.search(r'Unit:\s*"(\w+)"', header1)
    unit = unit_match.group(1) if unit_match else "mm"
    print('unit ',unit)
    unit_scale = {"mm": 1e-3, "m": 1.0, "cm": 1e-2}.get(unit, 1e-3)

    # data starts after the 3rd line (header1, header2, column-label line)
    data_lines = lines[2:]
    # replace Nan/NaN with 'nan' so numpy can parse it
    cleaned = "\n".join(
        re.sub(r"\bNan\b", "nan", line, flags=re.IGNORECASE) for line in data_lines
    )
    data = np.fromstring(cleaned, sep=" ")
    print('data max = ',data.max())
    data = data.reshape(-1, 6)  # X, Y, Z, Jx, Jy, Jz
    print('data max = ',data.max())
    data[:, :3] = data[:, :3]*1000
    data[:, :3] = np.rint(data[:, :3])
    print('data max = ',data.max())
 
    nx = int(round((grid_max[0] - grid_min[0]) / grid_size[0])) + 1
    ny = int(round((grid_max[1] - grid_min[1]) / grid_size[1])) + 1
    nz = int(round((grid_max[2] - grid_min[2]) / grid_size[2])) + 1
    
    print(nx,ny,nz)
 
    expected = nx * ny * nz
    if data.shape[0] != expected:
        raise ValueError(
            f"Parsed {data.shape[0]} rows but expected {expected} "
            f"({nx} x {ny} x {nz}) -- check ordering/header parsing."
        )
    print('data max = ',data.max())
    J = data[:, 3:6].reshape(nx, ny, nz, 3, order='F')
    print('J max = ',J.max())
    J_flat = np.column_stack([
        J[...,0].ravel(order='F'),
        J[...,1].ravel(order='F'),
        J[...,2].ravel(order='F')
    ])
 
    Jx = J[..., 0]
    Jy = J[..., 1]
    Jz = J[..., 2]
 
    xs = (grid_min[0] + np.arange(nx) * grid_size[0]) * unit_scale
    ys = (grid_min[1] + np.arange(ny) * grid_size[1]) * unit_scale
    zs = (grid_min[2] + np.arange(nz) * grid_size[2]) * unit_scale
    spacing = grid_size * unit_scale  # (dx, dy, dz) in meters
 
    return (xs, ys, zs), tuple(spacing), (Jx, Jy, Jz)



In [96]:
df = pd.DataFrame(data)

In [105]:
df

,0,1,2,3,4,5
0,-52.0,-111.0,-51.0,NaN,NaN,NaN
1,-52.0,-111.0,-50.0,NaN,NaN,NaN
2,-52.0,-111.0,-49.0,NaN,NaN,NaN
3,-52.0,-111.0,-48.0,NaN,NaN,NaN
4,-52.0,-111.0,-47.0,NaN,NaN,NaN
...,...,...,...,...,...,...
92,-52.0,-111.0,41.0,NaN,NaN,NaN
93,-52.0,-111.0,42.0,NaN,NaN,NaN
94,-52.0,-111.0,43.0,NaN,NaN,NaN
95,-52.0,-111.0,44.0,NaN,NaN,NaN


In [104]:
filtered_A = df.loc[df[3].notna()]
print(filtered_A)

Empty DataFrame
Columns: [0, 1, 2, 3, 4, 5]
Index: []


ValueError: cannot reshape array of size 291 into shape (150,166,124,3)

In [92]:
print(J_flat[:20])
print(J_flat[-20:])
print(J_flat.min(), J_flat.max())

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
-1108.405953300136 596.2138591062898


In [121]:
# ---------------------------------------------------------------------
# 2. Biot-Savart kernel K(r) = r / |r|^3 (double-sized for linear conv)
# ---------------------------------------------------------------------
def build_kernel(grid_shape, spacing):
    nx, ny, nz = grid_shape
    dx, dy, dz = spacing
 
    kx = (np.arange(2 * nx - 1) - (nx - 1)) * dx
    ky = (np.arange(2 * ny - 1) - (ny - 1)) * dy
    kz = (np.arange(2 * nz - 1) - (nz - 1)) * dz
 
    Kx, Ky, Kz = np.meshgrid(kx, ky, kz, indexing="ij")
    r2 = Kx**2 + Ky**2 + Kz**2
    r2[r2 == 0] = np.inf
    r3 = r2 * np.sqrt(r2)
 
    return Kx / r3, Ky / r3, Kz / r3
 
 
def fft_convolve_full(a, b):
    """Linear (non-circular) 3D convolution via zero-padded FFT."""
    out_shape = np.array(a.shape) + np.array(b.shape) - 1
    # pad each axis up to a fast FFT length (power of 2 here for simplicity)
    fshape = [int(2 ** np.ceil(np.log2(s))) for s in out_shape]
 
    A = np.fft.rfftn(a, fshape)
    B = np.fft.rfftn(b, fshape)
    conv = np.fft.irfftn(A * B, fshape)
 
    slices = tuple(slice(0, s) for s in out_shape)
    return conv[slices]

In [122]:
# ---------------------------------------------------------------------
# 3. Compute B from the 6 cross-product convolutions
# ---------------------------------------------------------------------
def compute_B(J_components, kernel, spacing, grid_shape):
    Jx, Jy, Jz = J_components
    Rx, Ry, Rz = kernel
    dx, dy, dz = spacing
    dV = dx * dy * dz
    nx, ny, nz = grid_shape
 
    JyKz = fft_convolve_full(Jy, Rz)
    JzKy = fft_convolve_full(Jz, Ry)
    JzKx = fft_convolve_full(Jz, Rx)
    JxKz = fft_convolve_full(Jx, Rz)
    JxKy = fft_convolve_full(Jx, Ry)
    JyKx = fft_convolve_full(Jy, Rx)
 
    Bx_full = JyKz - JzKy
    By_full = JzKx - JxKz
    Bz_full = JxKy - JyKx
 
    start = (nx - 1, ny - 1, nz - 1)
    sl = tuple(slice(start[i], start[i] + grid_shape[i]) for i in range(3))
 
    Bx = MU0 / (4 * np.pi) * Bx_full[sl] * dV
    By = MU0 / (4 * np.pi) * By_full[sl] * dV
    Bz = MU0 / (4 * np.pi) * Bz_full[sl] * dV
    return Bx, By, Bz

In [108]:
#print(Jx.min(), Jx.max())
#print(Jy.min(), Jy.max())
#print(Jz.min(), Jz.max())

J_flat = np.column_stack([Jx.ravel(), Jy.ravel(), Jz.ravel()])
print(J_flat[:20])
print(J_flat[-20:])
print(J_flat.min(), J_flat.max())

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
-1108.405953300136 596.2138591062898


In [5]:
# ---------------------------------------------------------------------
# Main driver
# ---------------------------------------------------------------------
if __name__ == "__main__":
    filepath = "E:/DBS_ansys_2a&3a.fld"
 
    (xs, ys, zs), spacing, J_components = parse_fld(filepath)
    grid_shape = (240, 300, 240)
    print("Grid shape:", grid_shape, "-- spacing (m):", spacing)

    kernel = build_kernel(grid_shape, spacing)
    Bx, By, Bz = compute_B(J_components, kernel, spacing, grid_shape)

    cx, cy, cz = grid_shape[0] // 2, grid_shape[1] // 2, grid_shape[2] // 2
    print("Sample B at grid center (T):", Bx[cx, cy, cz], By[cx, cy, cz], Bz[cx, cy, cz])

    np.savez("B_field_result.npz", xs=xs, ys=ys, zs=zs, Bx=Bx, By=By, Bz=Bz)
    print("Saved B_field_result.npz")

total lines in file= 3087602
last line= 9.6999999999999989e-02 5.4000000000000006e-02 7.0999999999999994e-02  Nan Nan Nan

length of data: 3087600
first data line =  [-0.052 -0.111 -0.052    nan    nan    nan]
last data line =  [0.097 0.054 0.071   nan   nan   nan]
[[ -52. -111.  -52.   nan   nan   nan]
 [ -52. -111.  -51.   nan   nan   nan]
 [ -52. -111.  -50.   nan   nan   nan]
 [ -52. -111.  -49.   nan   nan   nan]
 [ -52. -111.  -48.   nan   nan   nan]
 [ -52. -111.  -47.   nan   nan   nan]
 [ -52. -111.  -46.   nan   nan   nan]
 [ -52. -111.  -45.   nan   nan   nan]
 [ -52. -111.  -44.   nan   nan   nan]
 [ -52. -111.  -43.   nan   nan   nan]
 [ -52. -111.  -42.   nan   nan   nan]
 [ -52. -111.  -41.   nan   nan   nan]
 [ -52. -111.  -40.   nan   nan   nan]
 [ -52. -111.  -39.   nan   nan   nan]
 [ -52. -111.  -38.   nan   nan   nan]
 [ -52. -111.  -37.   nan   nan   nan]
 [ -52. -111.  -36.   nan   nan   nan]
 [ -52. -111.  -35.   nan   nan   nan]
 [ -52. -111.  -34.   nan   nan 

C:\Users\nebula\AppData\Local\Temp\ipykernel_13812\1976606041.py:101: DeprecationWarning: `axes` should not be `None` if `s` is not `None` (Deprecated in NumPy 2.0). In a future version of NumPy, this will raise an error and `s[i]` will correspond to the size along the transformed axis specified by `axes[i]`. To retain current behaviour, pass a sequence [0, ..., k-1] to `axes` for an array of dimension k.
  A = np.fft.rfftn(a, fshape)
C:\Users\nebula\AppData\Local\Temp\ipykernel_13812\1976606041.py:102: DeprecationWarning: `axes` should not be `None` if `s` is not `None` (Deprecated in NumPy 2.0). In a future version of NumPy, this will raise an error and `s[i]` will correspond to the size along the transformed axis specified by `axes[i]`. To retain current behaviour, pass a sequence [0, ..., k-1] to `axes` for an array of dimension k.
  B = np.fft.rfftn(b, fshape)
C:\Users\nebula\AppData\Local\Temp\ipykernel_13812\1976606041.py:103: DeprecationWarning: `axes` should not be `None` if `

MemoryError: Unable to allocate 8.02 GiB for an array with shape (1024, 1024, 513) and data type complex128

In [126]:
MU0 = 4 * np.pi * 1e-7

filepath = "ansys_use.fld"
(xs, ys, zs), spacing, (Jx, Jy, Jz) = parse_fld(filepath)
dx, dy, dz = spacing
dV = dx*dy*dz
nx, ny, nz = Jx.shape


Xg, Yg, Zg = np.meshgrid(xs, ys, zs, indexing="ij")
r_grid = np.column_stack([
    Xg.ravel(),
    Yg.ravel(),
    Zg.ravel()
])

J_flat = np.column_stack([
    Jx.ravel(),
    Jy.ravel(),
    Jz.ravel()
])

total lines in file= 3087602
line 1779117 3.4000000000000002e-02 -3.9999999999999994e-02 3.5000000000000003e-02  -2.5657783130450168e-02 2.3512267866146873e-02 -1.9431091947852354e-03

unit  mm
data max =  nan
data max =  nan
data max =  nan
150 166 124
data max =  nan
J max =  nan


In [118]:
Jx.max()

np.float64(nan)

In [116]:
J_flat.max()

np.float64(nan)

In [69]:
raw = """-0.1066 0.0464 -0.0604
-0.1020 0.0631 -0.0256
-0.1085 0.0302 -0.0266
-0.1099 0.0131 -0.0627
-0.1074 0.0329 0.0080
-0.0989 0.0403 0.0413
-0.1011 0.0044 0.0408
-0.1083 -0.0011 0.0071
-0.0861 0.0988 0.0090
-0.0887 0.0757 0.0412
-0.0702 0.0758 0.0707
-0.1003 0.0659 0.0081
-0.0808 0.0413 0.0720
-0.0526 0.0406 0.0952
-0.0537 0.0059 0.0969
-0.0829 0.0062 0.0728
-0.0637 0.1254 0.0136
-0.0332 0.1397 0.0174
-0.0337 0.1274 0.0485
-0.0672 0.1089 0.0443
-0.0358 0.1048 0.0750
0.0001 0.0775 0.0967
-0.0184 0.0440 0.1063
-0.0368 0.0753 0.0922
-0.0185 0.0105 0.1096
0.0186 0.0105 0.1096
0.0186 -0.0233 0.1059
-0.0185 -0.0237 0.1058
0.0001 0.1445 0.0187
0.0001 0.1316 0.0500
0.0331 0.1397 0.0173
0.0638 0.1253 0.0135
0.0671 0.1088 0.0444
0.0338 0.1273 0.0486
0.0001 0.1093 0.0771
0.0358 0.1048 0.0750
0.0368 0.0752 0.0923
0.0184 0.0442 0.1062
0.0525 0.0406 0.0953
0.0809 0.0413 0.0721
0.0828 0.0061 0.0728
0.0535 0.0062 0.0970
0.0862 0.0986 0.0089
0.1003 0.0660 0.0082
0.0887 0.0757 0.0412
0.0699 0.0758 0.0709
0.0989 0.0404 0.0413
0.1074 0.0329 0.0081
0.1083 -0.0011 0.0068
0.1010 0.0044 0.0410
0.1020 0.0630 -0.0260
0.1065 0.0469 -0.0600
0.1098 0.0131 -0.0622
0.1083 0.0301 -0.0262
-0.1088 -0.0032 -0.0284
-0.1017 -0.0360 -0.0281
-0.0951 -0.0524 -0.0623
-0.1068 -0.0205 -0.0625
-0.1017 -0.0339 0.0056
-0.0952 -0.0308 0.0391
-0.0781 -0.0628 0.0394
-0.0866 -0.0640 0.0055
-0.0758 -0.0797 -0.0621
-0.0861 -0.0660 -0.0282
-0.0632 -0.0905 -0.0278
-0.0489 -0.0994 -0.0621
-0.0786 -0.0287 0.0696
-0.0518 -0.0277 0.0927
-0.0181 -0.0542 0.0923
-0.0552 -0.0627 0.0707
-0.0513 -0.0861 0.0397
-0.0335 -0.1033 0.0062
-0.0331 -0.1051 -0.0278
-0.0636 -0.0884 0.0060
-0.0186 -0.0801 0.0690
0.0186 -0.0802 0.0689
0.0169 -0.0972 0.0397
-0.0170 -0.0972 0.0397
0.0000 -0.1086 0.0070
0.0000 -0.1106 -0.0275
0.0170 -0.1098 -0.0618
-0.0171 -0.1098 -0.0619
0.0517 -0.0276 0.0928
0.0786 -0.0284 0.0698
0.0553 -0.0628 0.0706
0.0182 -0.0542 0.0922
0.0513 -0.0861 0.0397
0.0637 -0.0884 0.0062
0.0330 -0.1051 -0.0277
0.0333 -0.1034 0.0063
0.0952 -0.0306 0.0392
0.1017 -0.0338 0.0058
0.0866 -0.0639 0.0056
0.0781 -0.0629 0.0394
0.0630 -0.0906 -0.0276
0.0861 -0.0661 -0.0280
0.0757 -0.0798 -0.0620
0.0488 -0.0994 -0.0621
0.1087 -0.0033 -0.0280
0.1068 -0.0206 -0.0621
0.0951 -0.0524 -0.0620
0.1017 -0.0361 -0.0278
"""

In [70]:
J_flat = np.column_stack([
    Jx.ravel(),
    Jy.ravel(),
    Jz.ravel()
])

X, Y, Z, = np.meshgrid(xs, ys, zs, indexing='ij')
r_grid = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
dx, dy, dz = spacing
dV = dx * dy * dz

B_sensors = biot_savart_chunked(sensors, r_grid, J_flat, dV)

In [71]:
import numpy as np

MU0 = 4 * np.pi * 1e-7

def biot_savart_chunked(sensors, r_grid, J_flat, dV, chunk_size=200000):
    B = np.zeros((len(sensors), 3))

    N = len(r_grid)

    for i, r in enumerate(sensors):
        Bi = np.zeros(3)

        for start in range(0, N, chunk_size):
            end = min(start + chunk_size, N)

            R = r - r_grid[start:end]
            R2 = np.sum(R**2, axis=1)
            mask = R2>0

            R = R[mask]
            J = J_flat[start:end][mask]
            R2 = R2[mask]
            R3 = R2 * np.sqrt(R2)

            Bi +- np.sum(np.cross(J, R) / R3[:, None], axis=0)

        B[i] = MU0 / (4*np.pi) * Bi * dV
    return B

In [72]:
#finding b at a specific point
B_sensors = biot_savart_chunked(sensors, r_grid, J_flat, dV)

i = 100 #any index
print("sensor location:", sensors[i])
print("B-field there:", B_sensors[i])
B_mag = np.linalg.norm(B_sensors, axis=1)
print(B_mag[i])

sensor location: [ 0.0951 -0.0524 -0.062 ]
B-field there: [0. 0. 0.]
0.0


In [73]:
print(r_grid.min(axis=0))
print(r_grid.max(axis=0))

print(sensors.min(axis=0))
print(sensors.max(axis=0))

[-0.052 -0.111 -0.052]
[0.097 0.054 0.071]
[-0.1099 -0.1106 -0.0627]
[0.1098 0.1445 0.1096]


In [117]:
print(J_flat.max())


nan


In [ ]:
x = np.linspace(-0.20, 0.20, 120)
y = np.linspace(-0.20, 0.20, 120)
z = np.linspace(-0.20, 0.20, 120)

X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
r_big = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])

J_big = np.zeros((len(r_big), 3))

In [49]:
print(J_flat.shape)
print("J min:", J_flat.min())
print("J max:", J_flat.max())
print("first 5", J_flat[:5])

(3087600, 3)
J min: -1108.405953300136
J max: 596.2138591062898
first 5 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [51]:
(len(r_grid), 3)

import re
import numpy as np
with open("", "r") as f:
    raw_fld = f.read()
num

(3087600, 3)

In [27]:
import inspect
import numpy as np

for name, obj in globals().items():
    if callable(obj):
        src = inspect.getsource(obj)
        if "fromstring" in src:
            print("found in :", name)
            print(src)
            

TypeError: module, class, method, function, traceback, frame, or code object was expected, got ZMQExitAutocall

In [20]:
import re
rows = []
for line in raw.splitlines():
    nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", line)
    
    if len(nums)>=3:
        rows.append([float(nums[0]), float(nums[1]), float(nums[2])])

sensors = np.array(rows)
print(sensors.shape)


(102, 3)


In [18]:

print(repr(raw))

'-0.1066 0.0464 -0.0604\n-0.1020 0.0631 -0.0256\n-0.1085 0.0302 -0.0266\n-0.1099 0.0131 -0.0627\n-0.1074 0.0329 0.0080\n-0.0989 0.0403 0.0413\n-0.1011 0.0044 0.0408\n-0.1083 -0.0011 0.0071\n-0.0861 0.0988 0.0090\n-0.0887 0.0757 0.0412\n-0.0702 0.0758 0.0707\n-0.1003 0.0659 0.0081\n-0.0808 0.0413 0.0720\n-0.0526 0.0406 0.0952\n-0.0537 0.0059 0.0969\n-0.0829 0.0062 0.0728\n-0.0637 0.1254 0.0136\n-0.0332 0.1397 0.0174\n-0.0337 0.1274 0.0485\n-0.0672 0.1089 0.0443\n-0.0358 0.1048 0.0750\n0.0001 0.0775 0.0967\n-0.0184 0.0440 0.1063\n-0.0368 0.0753 0.0922\n-0.0185 0.0105 0.1096\n0.0186 0.0105 0.1096\n0.0186 -0.0233 0.1059\n-0.0185 -0.0237 0.1058\n0.0001 0.1445 0.0187\n0.0001 0.1316 0.0500\n0.0331 0.1397 0.0173\n0.0638 0.1253 0.0135\n0.0671 0.1088 0.0444\n0.0338 0.1273 0.0486\n0.0001 0.1093 0.0771\n0.0358 0.1048 0.0750\n0.0368 0.0752 0.0923\n0.0184 0.0442 0.1062\n0.0525 0.0406 0.0953\n0.0809 0.0413 0.0721\n0.0828 0.0061 0.0728\n0.0535 0.0062 0.0970\n0.0862 0.0986 0.0089\n0.1003 0.0660 0.0082\

In [ ]:
filepath = "E:/DBS_ansys_2a&3a.fld"
(xs, ys, zs), spacing, J_components = parse_fld(filepath)

In [ ]:
kernel = build_kernel(grid_shape, spacing)
Bx, By, Bz = compute_B(J_components, kernel, spacing, grid_shape)

In [ ]:
cx, cy, cz = grid_shape[0] // 2, grid_shape[1] // 2, grid_shape[2] // 2
print("Sample B at grid center (T):", Bx[cx, cy, cz], By[cx, cy, cz], Bz[cx, cy, cz])
np.savez("B_field_result.npz", xs=xs, ys=ys, zs=zs, Bx=Bx, By=By, Bz=Bz)
print("Saved B_field_result.npz")

In [ ]:
# measuring at coordinates

import numpy as np

raw = """
-0.1066 0.0464 -0.0604
-0.1020 0.0631 -0.0256
-0.1085 0.0302 -0.0266
-0.1099 0.0131 -0.0627
-0.1074 0.0329 0.0080
-0.0989 0.0403 0.0413
-0.1011 0.0044 0.0408
-0.1083 -0.0011 0.0071
-0.0861 0.0988 0.0090
-0.0887 0.0757 0.0412
-0.0702 0.0758 0.0707
-0.1003 0.0659 0.0081
-0.0808 0.0413 0.0720
-0.0526 0.0406 0.0952
-0.0537 0.0059 0.0969
-0.0829 0.0062 0.0728
-0.0637 0.1254 0.0136
-0.0332 0.1397 0.0174
-0.0337 0.1274 0.0485
-0.0672 0.1089 0.0443
-0.0358 0.1048 0.0750
0.0001 0.0775 0.0967
-0.0184 0.0440 0.1063
-0.0368 0.0753 0.0922
-0.0185 0.0105 0.1096
0.0186 0.0105 0.1096
0.0186 -0.0233 0.1059
-0.0185 -0.0237 0.1058
0.0001 0.1445 0.0187
0.0001 0.1316 0.0500
0.0331 0.1397 0.0173
0.0638 0.1253 0.0135
0.0671 0.1088 0.0444
0.0338 0.1273 0.0486
0.0001 0.1093 0.0771
0.0358 0.1048 0.0750
0.0368 0.0752 0.0923
0.0184 0.0442 0.1062
0.0525 0.0406 0.0953
0.0809 0.0413 0.0721
0.0828 0.0061 0.0728
0.0535 0.0062 0.0970
0.0862 0.0986 0.0089
0.1003 0.0660 0.0082
0.0887 0.0757 0.0412
0.0699 0.0758 0.0709
0.0989 0.0404 0.0413
0.1074 0.0329 0.0081
0.1083 -0.0011 0.0068
0.1010 0.0044 0.0410
0.1020 0.0630 -0.0260
0.1065 0.0469 -0.0600
0.1098 0.0131 -0.0622
0.1083 0.0301 -0.0262
-0.1088 -0.0032 -0.0284
-0.1017 -0.0360 -0.0281
-0.0951 -0.0524 -0.0623
-0.1068 -0.0205 -0.0625
-0.1017 -0.0339 0.0056
-0.0952 -0.0308 0.0391
-0.0781 -0.0628 0.0394
-0.0866 -0.0640 0.0055
-0.0758 -0.0797 -0.0621
-0.0861 -0.0660 -0.0282
-0.0632 -0.0905 -0.0278
-0.0489 -0.0994 -0.0621
-0.0786 -0.0287 0.0696
-0.0518 -0.0277 0.0927
-0.0181 -0.0542 0.0923
-0.0552 -0.0627 0.0707
-0.0513 -0.0861 0.0397
-0.0335 -0.1033 0.0062
-0.0331 -0.1051 -0.0278
-0.0636 -0.0884 0.0060
-0.0186 -0.0801 0.0690
0.0186 -0.0802 0.0689
0.0169 -0.0972 0.0397
-0.0170 -0.0972 0.0397
0.0000 -0.1086 0.0070
0.0000 -0.1106 -0.0275
0.0170 -0.1098 -0.0618
-0.0171 -0.1098 -0.0619
0.0517 -0.0276 0.0928
0.0786 -0.0284 0.0698
0.0553 -0.0628 0.0706
0.0182 -0.0542 0.0922
0.0513 -0.0861 0.0397
0.0637 -0.0884 0.0062
0.0330 -0.1051 -0.0277
0.0333 -0.1034 0.0063
0.0952 -0.0306 0.0392
0.1017 -0.0338 0.0058
0.0866 -0.0639 0.0056
0.0781 -0.0629 0.0394
0.0630 -0.0906 -0.0276
0.0861 -0.0661 -0.0280
0.0757 -0.0798 -0.0620
0.0488 -0.0994 -0.0621
0.1087 -0.0033 -0.0280
0.1068 -0.0206 -0.0621
0.0951 -0.0524 -0.0620
0.1017 -0.0361 -0.0278

"""

sensors = np.fromstring(raw, sep=" ").reshape(-1, 3)

from scipy.interpolate import RegularGridInterpolator

interp_Bx = RegularGridInterpolator((xs, ys, zs), Bx)
interp_By = RegularGridInterpolator((xs, ys, zs), By)
interp_Bz = RegularGridInterpolator((xs, ys, zs), Bz)


Bx_s = interp_Bx(sensors)
By_s = interp_By(sensors)
Bz_s = interp_Bz(sensors)

B_sensors = np.column_stack([Bx_s, By_s, Bz_s])


print(xs.min(), xs.max())
print(ys.min(), ys.max())
print(zs.min(), zs.max())
print(spacing)